# S1 · Pydantic AI y tools de lectura
Cada bloque empieza con un **ejemplo mínimo** (lo que viste en la demo) y sigue con un **ejercicio** corto que lo varía. Son 5 ejercicios y un reto (≈ 60 min).
Corre sin red ni LLM con el modelo simulado (`COURSE_OFFLINE=1`, por defecto). Para IA real: `COURSE_OFFLINE=0` en `.env` y *Restart* del kernel.


## Preparación
`preparar()` lee tu `.env`; `crear_modelo` devuelve el modelo de tu ruta: **Ollama** (`gemma4:e2b`), **Gemini** o el **simulado** (no es IA: solo sirve para ver el recorrido sin red ni cuota).


In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.capabilities import Capability

from utilidades.config import crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar, ver_mensajes

settings, OFFLINE = preparar()
modelo = crear_modelo(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE))


def tools_usadas(resultado) -> set[str]:
    """Nombres de las tools que el modelo pidió en un run (para autocorregir)."""
    return {
        p.tool_name
        for m in resultado.all_messages()
        for p in m.parts
        if p.part_kind == "tool-call" and p.tool_name != "final_result"
    }


## Ejemplo 1 · Un agente y una salida tipada (Demo 1)
Un `Agent` es un **modelo** más unas **instrucciones**. `await agente.run(pedido)` ejecuta el ciclo y `r.output` es la respuesta.
Con `output_type=Contacto` la salida deja de ser texto: es un objeto de Pydantic **validado**.


In [ ]:
agente = Agent(modelo, instructions="Eres un asistente de sociología. Responde en dos frases.")
r = await agente.run("¿Qué es la sociología digital?")
print(r.output)


class Contacto(BaseModel):
    nombre: str
    ciudad: str
    edad: int = Field(ge=0, description="Edad en años")


extractor = Agent(modelo, instructions="Extrae los datos del texto.", output_type=Contacto)
r = await extractor.run("Ana Quispe tiene 34 años y vive en Cochabamba.")
print(r.output)


## Ejercicio 1 · Extrae un libro
**Se apoya en:** Demo 1 · **Nivel:** básico · ≈ 8 min
1. Completa la clase `Libro` con tres campos: `titulo` (`str`), `autor` (`str`) y `anio` (`int`, año de publicación).
2. Ejecuta: el agente debe devolver un `Libro`, no un texto.


In [ ]:
class Libro(BaseModel):
    # TODO Ejercicio 1: añade titulo (str), autor (str) y anio (int).
    pass


lector = Agent(modelo, instructions="Extrae los datos del libro del texto.", output_type=Libro)
r = await lector.run("«Raza de bronce» es una novela que Alcides Arguedas publicó en 1919.")
print(r.output)
revisar("Ejercicio 1 · Libro tiene titulo, autor y anio", lambda: {"titulo", "autor", "anio"} <= set(Libro.model_fields), "faltan campos en Libro")
revisar("Ejercicio 1 · anio es un número", lambda: isinstance(r.output.anio, int), "anio debe ser int")


## Ejemplo 2 · Una tool directa en el agente (Demo 2)
Una **tool** es una función de Python que el modelo puede **pedir** ejecutar. El modelo no ejecuta nada: elige la tool y sus argumentos; Pydantic AI llama a tu función y le devuelve el resultado.
El **docstring** y los tipos de los parámetros son lo que el modelo lee para decidir.


In [ ]:
demografo = Agent(modelo, instructions="Usa tus herramientas. No inventes datos.")


@demografo.tool_plain
def consultar_poblacion(anio: int) -> int | None:
    """Población (ficticia) del municipio en un año de censo: 2012 o 2024."""
    return {2012: 1000, 2024: 1200}.get(anio)


r = await demografo.run("¿Cuál era la población en el censo 2024?")
print(r.output)
ver_mensajes(r)  # mira la fila «🤖 pide tool»: ahí decide el modelo


## Ejercicio 2 · Tu primera tool
**Se apoya en:** Demo 2 · **Nivel:** básico · ≈ 8 min
Añade al agente `vendedor` una tool `precio_producto(nombre: str) -> float | None` que busque el precio en Bs. en el diccionario `PRECIOS` (en minúsculas). Escribe un buen docstring.


In [ ]:
PRECIOS = {"cuaderno": 12.5, "mochila": 180.0, "calculadora": 95.0}  # Bs., ficticios

vendedor = Agent(modelo, instructions="Respondes precios de la librería. Usa tus herramientas; no inventes precios.")

# TODO Ejercicio 2: decora con @vendedor.tool_plain una función precio_producto(nombre: str) -> float | None
# que devuelva PRECIOS.get(nombre.lower()). No olvides el docstring.
r = await vendedor.run("¿Cuánto cuesta una mochila?")
print(r.output)
ver_mensajes(r)
revisar("Ejercicio 2 · el modelo pidió precio_producto", lambda: "precio_producto" in tools_usadas(r), "¿definiste la tool con @vendedor.tool_plain?")


## Ejemplo 3 · Una Capability con su tool y un informe tipado (Demo 3)
Una **Capability** agrupa tools **e** instrucciones en una pieza reutilizable: la defines una vez y se la das a cualquier agente con `capabilities=[...]`.
Junto con `output_type=Informe`, el agente consulta la tool y entrega un objeto validado.


In [ ]:
censos = Capability(
    id="censos",
    description="Consulta datos de población.",
    instructions="Usa consultar_poblacion_censo para obtener cifras. No inventes datos.",
)


@censos.tool_plain
def consultar_poblacion_censo(anio: int) -> int | None:
    """Población (ficticia) del municipio en un año de censo: 2012 o 2024."""
    return {2012: 1000, 2024: 1200}.get(anio)


class Informe(BaseModel):
    poblacion_2012: int = Field(ge=0)
    poblacion_2024: int = Field(ge=0)
    crecimiento_pct: float
    explicacion: str


analista = Agent(modelo, capabilities=[censos], output_type=Informe)
r = await analista.run("Compara la población de 2012 y 2024 y calcula el crecimiento en porcentaje.")
print(r.output)
ver_mensajes(r)


**Ojo:** Pydantic valida la **forma** (tipos, `ge=0`), no que la cuenta sea correcta. Comprueba tú: (1200 − 1000) / 1000 = 20 %.


## Ejemplo 4 · Una tool que lee una API abierta (Demo 4)
La tool hace un `GET` con `httpx` a una API pública **sin clave** (Banco Mundial) y devuelve solo los campos útiles. `async def` porque espera a la red.


In [ ]:
import httpx

apis = Capability(id="apis_publicas", description="Estadísticas de países (Banco Mundial).")


@apis.tool_plain
async def poblacion_pais(pais: str, anio: int) -> dict:
    """Población total de un país en un año, según el Banco Mundial.

    Args:
        pais: código ISO3, como BOL, PER o ARG.
        anio: año, por ejemplo 2023.
    """
    url = f"https://api.worldbank.org/v2/country/{pais.upper()}/indicator/SP.POP.TOTL"
    async with httpx.AsyncClient(timeout=15) as cliente:
        respuesta = await cliente.get(url, params={"date": anio, "format": "json"})
    datos = respuesta.json()
    if len(datos) < 2 or not datos[1] or datos[1][0]["value"] is None:
        return {"error": f"No hay dato para {pais} en {anio}"}
    return {"pais": datos[1][0]["country"]["value"], "anio": anio, "poblacion": datos[1][0]["value"]}


print(await poblacion_pais("BOL", 2023))  # puedes probar la función sola, sin modelo
estadistico = Agent(modelo, instructions="Usa tus herramientas. No inventes datos.", capabilities=[apis])
r = await estadistico.run("¿Cuál era la población de Bolivia en 2023?")
print(r.output)
ver_mensajes(r)


## Ejercicio 3 · Una Capability de clima con Open-Meteo
**Se apoya en:** Demo 4 · **Nivel:** medio · ≈ 12 min
Escribe en la Capability `clima` la tool `consultar_clima(ciudad: str) -> dict`:
1. Busca las coordenadas en `COORDENADAS`; si la ciudad no está, devuelve `{"error": "..."}` con las ciudades disponibles (un error amable también es una respuesta útil para el modelo).
2. Haz `GET https://api.open-meteo.com/v1/forecast` con `latitude`, `longitude`, `current="temperature_2m,wind_speed_10m"` y `timezone="America/La_Paz"`.
3. Devuelve `{"ciudad", "temperatura_c", "viento_kmh"}` desde `respuesta.json()["current"]`.


In [ ]:
COORDENADAS = {"la paz": (-16.50, -68.15), "cochabamba": (-17.39, -66.16), "santa cruz": (-17.78, -63.18)}

clima = Capability(
    id="clima",
    description="Clima actual de ciudades de Bolivia.",
    instructions="Usa consultar_clima para hablar del clima. No inventes temperaturas.",
)

# TODO Ejercicio 3: decora con @clima.tool_plain una función async consultar_clima(ciudad: str) -> dict
# (mira poblacion_pais del ejemplo 4: misma forma, otra URL y otros parámetros).
try:  # primero la función sola, sin modelo
    prueba, desconocida = await consultar_clima("La Paz"), await consultar_clima("Tarija")
except NameError:
    prueba = desconocida = None
print(prueba, desconocida)
revisar("Ejercicio 3 · consultar_clima devuelve temperatura_c", lambda: "temperatura_c" in prueba, "devuelve un dict con temperatura_c")
revisar("Ejercicio 3 · ciudad desconocida → error amable", lambda: "error" in desconocida, "si la ciudad no está, devuelve {'error': ...}")
meteorologo = Agent(modelo, capabilities=[clima])
r = await meteorologo.run("¿Qué clima hace ahora en Santa Cruz?")
print(r.output)
revisar("Ejercicio 3 · el modelo pidió consultar_clima", lambda: "consultar_clima" in tools_usadas(r), "¿la tool está en la Capability clima?")


## Ejemplo 5 · Una tool que lee SQLite (Demo 5)
Creamos una base local ficticia y una tool de **solo lectura**: `?mode=ro` abre el archivo sin permiso de escritura, y los `?` del SQL son **parámetros** (nunca pegues el texto del usuario dentro del SQL). El modelo no escribe SQL: solo elige el argumento `texto`.


In [ ]:
import sqlite3
from pathlib import Path

DB_PROYECTOS = Path("proyectos.db")
with sqlite3.connect(DB_PROYECTOS) as conexion:
    conexion.execute("CREATE TABLE IF NOT EXISTS proyectos (nombre TEXT PRIMARY KEY, area TEXT, descripcion TEXT)")
    conexion.executemany(
        "INSERT OR REPLACE INTO proyectos VALUES (?, ?, ?)",
        [
            ("Censo Local", "Datos", "Explorador de indicadores demográficos"),
            ("Lenguas Vivas", "Educación", "Aplicación para aprender lenguas"),
            ("Mapa Ambiental", "Medio ambiente", "Visualizador de información ambiental"),
        ],
    )
conexion.close()

datos_locales = Capability(
    id="datos_locales",
    description="Proyectos guardados en una base SQLite local.",
    instructions="Usa buscar_proyectos. No inventes proyectos.",
)


@datos_locales.tool_plain
def buscar_proyectos(texto: str) -> list[dict]:
    """Busca proyectos por nombre o área temática (por ejemplo Datos, Ambiental o Censo)."""
    conexion = sqlite3.connect(DB_PROYECTOS.resolve().as_uri() + "?mode=ro", uri=True)
    conexion.row_factory = sqlite3.Row
    try:
        filas = conexion.execute(
            "SELECT nombre, area, descripcion FROM proyectos WHERE nombre LIKE ? OR area LIKE ?",
            (f"%{texto}%", f"%{texto}%"),
        ).fetchall()
    finally:
        conexion.close()
    return [dict(fila) for fila in filas]


print(buscar_proyectos("Datos"))
asistente = Agent(modelo, instructions="Usa tus herramientas. No inventes datos.", capabilities=[apis, datos_locales])
r = await asistente.run("Busca proyectos del área Datos.")
print(r.output)
ver_mensajes(r)


## Ejercicio 4 · Una biblioteca en SQLite
**Se apoya en:** Demo 5 · **Nivel:** medio · ≈ 12 min
La celda crea `biblioteca.db` con una tabla `libros(titulo, autor, anio, tema)`. Escribe en la Capability `biblioteca` la tool `buscar_libros(texto: str) -> list[dict]` que:
- abra la base **solo lectura** (`?mode=ro`),
- busque `texto` en `titulo`, `autor` **o** `tema` con `LIKE ?` (parámetros, no f-strings dentro del SQL),
- devuelva una lista de diccionarios.


In [ ]:
DB_BIBLIOTECA = Path("biblioteca.db")
with sqlite3.connect(DB_BIBLIOTECA) as conexion:
    conexion.execute("CREATE TABLE IF NOT EXISTS libros (titulo TEXT PRIMARY KEY, autor TEXT, anio INTEGER, tema TEXT)")
    conexion.executemany(
        "INSERT OR REPLACE INTO libros VALUES (?, ?, ?, ?)",
        [
            ("Raza de bronce", "Alcides Arguedas", 1919, "Bolivia"),
            ("Juan de la Rosa", "Nataniel Aguirre", 1885, "Bolivia"),
            ("Introducción a los agentes", "Ana Quispe", 2030, "Inteligencia artificial"),
            ("Datos abiertos para municipios", "Luis Mamani", 2028, "Datos"),
        ],
    )
conexion.close()

biblioteca = Capability(
    id="biblioteca",
    description="Catálogo de libros en una base SQLite local.",
    instructions="Usa buscar_libros para responder sobre libros. No inventes títulos.",
)

# TODO Ejercicio 4: decora con @biblioteca.tool_plain una función buscar_libros(texto: str) -> list[dict]
# (mira buscar_proyectos del ejemplo 5: misma forma, otra tabla y tres columnas en el WHERE).
revisar("Ejercicio 4 · buscar_libros('Bolivia') devuelve 2 libros", lambda: len(buscar_libros("Bolivia")) == 2, "busca en titulo, autor y tema")
revisar("Ejercicio 4 · busca también por autor", lambda: buscar_libros("Arguedas")[0]["titulo"] == "Raza de bronce", "incluye autor en el WHERE")
revisar("Ejercicio 4 · nada inventado si no hay coincidencias", lambda: buscar_libros("zzz") == [], "sin coincidencias, lista vacía")


## Ejercicio 5 · Un agente con dos capabilities
**Se apoya en:** Demo 5 · **Nivel:** medio · ≈ 10 min
Crea `bibliotecario` con `capabilities=[clima, biblioteca]` y escribe una `pregunta` que necesite **las dos** tools (por ejemplo, el clima de una ciudad y libros de un tema). Mira en `ver_mensajes` qué tools pidió el modelo y en qué orden.


In [ ]:
# TODO Ejercicio 5: crea el agente con las dos capabilities y escribe la pregunta.
bibliotecario = Agent(modelo, instructions="Usa tus herramientas. No inventes datos.")
pregunta = "Hola"
r = await bibliotecario.run(pregunta)
print(r.output)
ver_mensajes(r)
usadas = tools_usadas(r)
revisar("Ejercicio 5 · el modelo pidió consultar_clima", lambda: "consultar_clima" in usadas, "¿pasaste capabilities=[clima, biblioteca]? (el simulado elige solo una tool: pruébalo con IA real)")
revisar("Ejercicio 5 · el modelo pidió buscar_libros", lambda: "buscar_libros" in usadas, "¿tu pregunta pide libros? (el simulado elige solo una tool: pruébalo con IA real)")


## Reto (opcional) · Tipo de cambio
**Se apoya en:** Demos 3 y 4 · **Nivel:** reto
1. Crea una Capability `finanzas` con la tool `tipo_de_cambio(moneda: str) -> dict` que lea `https://open.er-api.com/v6/latest/USD` (sin clave; `respuesta.json()["rates"]` incluye `"BOB"`) y devuelva cuántas unidades de `moneda` vale 1 USD.
2. Úsala en un agente con `output_type=Conversion` (`monto_usd: float`, `monto_bob: float`) para «¿Cuántos bolivianos son 50 dólares?».
3. Extra: pon `defer_loading=True` en `clima` y `biblioteca` y mira en `ver_mensajes` cómo el modelo primero **carga** la capability que necesita.


In [ ]:
# TODO Reto (opcional): escribe aquí tu Capability finanzas y el agente con output_type=Conversion.
